# Create differently chunk IO.zarr

In [1]:
import xarray as xr
zarr_ds = xr.open_dataset(
    "gcs://nmfs_odp_nwfsc/CB/mind_the_chl_gap/IO.zarr",
    engine="zarr",
    backend_kwargs={"storage_options": {"token": "anon"}},
    consolidated=True,
    chunks={}
)

In [2]:
import numpy as np
all_nan_CHL = np.isnan(zarr_ds['CHL_cmes-level3']).all(dim=["lon", "lat"]).compute()  # find sample indices where CHL is NaN

zarr_ds = zarr_ds.sel(time=(~all_nan_CHL))  # select samples with CHL not NaN

zarr_ds = zarr_ds.sortby('time')


In [3]:
vars = ['CHL_cmes-level3', 'CHL_cmes-cloud', 'u_wind', 'v_wind', 'sst', 'air_temp']
zarr_ds = zarr_ds[vars]
zarr_ds

<xarray.Dataset> Size: 8GB
Dimensions:          (time: 9193, lat: 177, lon: 241)
Coordinates:
  * time             (time) datetime64[ns] 74kB 1997-10-01 ... 2022-12-31
  * lat              (lat) float32 708B 32.0 31.75 31.5 ... -11.5 -11.75 -12.0
  * lon              (lon) float32 964B 42.0 42.25 42.5 ... 101.5 101.8 102.0
Data variables:
    CHL_cmes-level3  (time, lat, lon) float32 2GB dask.array<chunksize=(99, 177, 241), meta=np.ndarray>
    CHL_cmes-cloud   (time, lat, lon) uint8 392MB dask.array<chunksize=(99, 177, 241), meta=np.ndarray>
    u_wind           (time, lat, lon) float32 2GB dask.array<chunksize=(99, 177, 241), meta=np.ndarray>
    v_wind           (time, lat, lon) float32 2GB dask.array<chunksize=(99, 177, 241), meta=np.ndarray>
    sst              (time, lat, lon) float32 2GB dask.array<chunksize=(99, 177, 241), meta=np.ndarray>
    air_temp         (time, lat, lon) float32 2GB dask.array<chunksize=(99, 177, 241), meta=np.ndarray>
Attributes: (12/92)
    Conventions:                     CF-1.8, ACDD-1.3
    DPM_reference:                   GC-UD-ACRI-PUG
    IODD_reference:                  GC-UD-ACRI-PUG
    acknowledgement:                 The Licensees will ensure that original ...
    citation:                        The Licensees will ensure that original ...
    cmems_product_id:                OCEANCOLOUR_GLO_BGC_L3_MY_009_103
    ...                              ...
    time_coverage_end:               2024-04-18T02:58:23Z
    time_coverage_resolution:        P1D
    time_coverage_start:             2024-04-16T21:12:05Z
    title:                           cmems_obs-oc_glo_bgc-plankton_my_l3-mult...
    westernmost_longitude:           -180.0
    westernmost_valid_longitude:     -180.0

In [4]:
import xarray as xr

# Desired test chunking; multiples of 8 for the UNet
chunks = {
    "time": 1,
    "lat": 40,
    "lon": 56,
}

# Rechunk the xarray/dask dataset
zarr_ds_chunked = zarr_ds.chunk(chunks)

# Clear inherited encoding from the source zarr
# This avoids stale compressor / chunks / preferred_chunks metadata issues
for name in zarr_ds_chunked.variables:
    zarr_ds_chunked[name].encoding.clear()

# Build encoding so variables are written with those zarr chunks
encoding = {}

for name, da in zarr_ds_chunked.data_vars.items():
    var_chunks = tuple(
        chunks[dim]
        for dim in da.dims
        if dim in chunks
    )

    if var_chunks:
        encoding[name] = {"chunks": var_chunks}

# Write to local zarr v2
filename = "/home/jovyan/shared-public/mindthegap/data/IO_rechunked.zarr"
zarr_ds_chunked.to_zarr(
    filename,
    mode="w",
    encoding=encoding,
    zarr_format=2,
)

In [10]:
import xarray as xr
filename = "/home/jovyan/shared-public/mindthegap/data/IO_rechunked.zarr"
ds = xr.open_zarr(filename, chunks={})
ds

<xarray.Dataset> Size: 8GB
Dimensions:          (time: 9193, lat: 177, lon: 241)
Coordinates:
  * time             (time) datetime64[ns] 74kB 1997-10-01 ... 2022-12-31
  * lat              (lat) float32 708B 32.0 31.75 31.5 ... -11.5 -11.75 -12.0
  * lon              (lon) float32 964B 42.0 42.25 42.5 ... 101.5 101.8 102.0
Data variables:
    air_temp         (time, lat, lon) float32 2GB dask.array<chunksize=(1, 40, 56), meta=np.ndarray>
    CHL_cmes-cloud   (time, lat, lon) uint8 392MB dask.array<chunksize=(1, 40, 56), meta=np.ndarray>
    CHL_cmes-level3  (time, lat, lon) float32 2GB dask.array<chunksize=(1, 40, 56), meta=np.ndarray>
    sst              (time, lat, lon) float32 2GB dask.array<chunksize=(1, 40, 56), meta=np.ndarray>
    u_wind           (time, lat, lon) float32 2GB dask.array<chunksize=(1, 40, 56), meta=np.ndarray>
    v_wind           (time, lat, lon) float32 2GB dask.array<chunksize=(1, 40, 56), meta=np.ndarray>
Attributes: (12/92)
    Conventions:                     CF-1.8, ACDD-1.3
    DPM_reference:                   GC-UD-ACRI-PUG
    IODD_reference:                  GC-UD-ACRI-PUG
    acknowledgement:                 The Licensees will ensure that original ...
    citation:                        The Licensees will ensure that original ...
    cmems_product_id:                OCEANCOLOUR_GLO_BGC_L3_MY_009_103
    ...                              ...
    time_coverage_end:               2024-04-18T02:58:23Z
    time_coverage_resolution:        P1D
    time_coverage_start:             2024-04-16T21:12:05Z
    title:                           cmems_obs-oc_glo_bgc-plankton_my_l3-mult...
    westernmost_longitude:           -180.0
    westernmost_valid_longitude:     -180.0

In [11]:
ds.chunks

Frozen({'time': (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [5]:
!pip install gcsfs

In [9]:
import fsspec

local_path = "/home/jovyan/shared-public/mindthegap/data/IO_rechunked.zarr"
gcs_path = "gcs://nmfs_odp_nwfsc/CB/mind_the_chl_gap/IO_rechunked.zarr"

fs_local = fsspec.filesystem("file")
fs_gcs = fsspec.filesystem(
    "gcs",
    token="/home/jovyan/.config/gcloud/application_default_credentials.json",
)

# Recursively copy local zarr directory to GCS
fs_gcs.put(
    local_path,
    gcs_path,
    recursive=True,
)

print("Zarr uploaded to GCS!")

Zarr uploaded to GCS!
